# Leave-one-out for a whole ridge path

For least squares and ridge regression, the leave-one-out error of row $i$ needs no refit:

$$
y_i - \hat{y}_{(-i)} = \frac{e_i}{1 - h_{ii}},
$$

where $e$ is the residual of the fit on all rows and $h_{ii}$ is the $i$-th diagonal entry of
$H_\lambda = X(X^\top X + \lambda I)^{-1}X^\top$. (The written problem "Leave-one-out from a
single least squares fit" derives it.) Here you turn that into code: the leverages, the
leave-one-out MSE for one $\lambda$, and then the whole path of $\lambda$ values from one
singular value decomposition. The data are 30 noisy points fit by a degree-12 polynomial,
with every coefficient, the intercept included, penalized.

**How to work through it.** Each part asks for one function. Replace the `TODO` and the
`raise NotImplementedError` with your code, then run the check cell under it. The check
prints `Part N passed` when your function is right, and stops with a message that says
what is wrong when it is not.

## Setup

Run this cell first.

In [ ]:
import numpy as np

rng = np.random.default_rng(3)
n = 30
x = np.sort(rng.uniform(-1, 1, n))
y = np.cos(3 * x) + 0.5 * x + rng.normal(0, 0.3, n)
X = np.vander(x, 13, increasing=True)          # columns 1, x, ..., x^12
lams = 10.0 ** np.arange(-6, 2)                 # 1e-06 to 10


def ridge_fit(X, y, lam):
    """Coefficients minimizing ||y - X b||^2 + lam ||b||^2."""
    gram = X.T @ X
    gram[np.diag_indices_from(gram)] += lam
    return np.linalg.solve(gram, X.T @ y)


def loo_by_refitting(X, y, lam):
    """Leave-one-out MSE the slow way: n separate fits."""
    errs = []
    for i in range(len(y)):
        keep = np.arange(len(y)) != i
        errs.append(y[i] - X[i] @ ridge_fit(X[keep], y[keep], lam))
    return float(np.mean(np.square(errs)))


print(X.shape, "slow leave-one-out MSE at lam = 1e-3:", round(loo_by_refitting(X, y, 1e-3), 5))

## Part 1 (coding task): Leverages

Write `leverages(X, lam)`, which returns the $n$ diagonal entries of
$H_\lambda = X(X^\top X + \lambda I)^{-1}X^\top$ as an array of shape `(n,)`.
Do it without building the $n \times n$ matrix: row $i$ contributes
$h_{ii} = x_i^\top (X^\top X + \lambda I)^{-1} x_i$.

In [ ]:
def leverages(X, lam):
    """Diagonal of X (X^T X + lam I)^{-1} X^T, shape (n,)."""
    # TODO
    raise NotImplementedError

In [ ]:
Z = np.random.default_rng(1).normal(size=(7, 3))
for lam in (0.0, 2.5):
    full = Z @ np.linalg.solve(Z.T @ Z + lam * np.eye(3), Z.T)
    got = np.asarray(leverages(Z, lam))
    assert got.shape == (7,), f"expected one leverage per row, shape (7,), got {got.shape}"
    assert np.allclose(got, np.diag(full)), f"lam = {lam}: the leverages differ from the diagonal of H"
assert np.isclose(leverages(Z, 0.0).sum(), 3), "with lam = 0 the leverages should add up to the number of columns"
h = leverages(X, 1e-3)
assert np.all((h > 0) & (h < 1)), "ridge leverages lie strictly between 0 and 1"
print("largest leverages:", np.round(np.sort(h)[-3:], 3), "  smallest:", np.round(np.sort(h)[:3], 3))
print("Part 1 passed")

## Part 2 (coding task): Leave-one-out MSE from one fit

Write `loo_mse(X, y, lam)`: fit ridge once on all rows, compute the residuals $e$ and the
leverages, and return the mean of $\big(e_i / (1 - h_{ii})\big)^2$. The check compares
it with `loo_by_refitting`.

In [ ]:
def loo_mse(X, y, lam):
    """Exact leave-one-out MSE of ridge with penalty lam, from a single fit."""
    # TODO
    raise NotImplementedError

In [ ]:
train = float(np.mean((y - X @ ridge_fit(X, y, 1e-3)) ** 2))
for lam in (1e-4, 1e-2, 1.0):
    fast, slow = loo_mse(X, y, lam), loo_by_refitting(X, y, lam)
    assert not np.isclose(fast, float(np.mean((y - X @ ridge_fit(X, y, lam)) ** 2))), (
        "that is the training MSE: divide each residual by 1 - h_ii before squaring")
    assert np.isclose(fast, slow, rtol=1e-6), f"lam = {lam}: from one fit {fast:.6f}, by refitting {slow:.6f}"
print(f"lam = 1e-3: training MSE {train:.4f}, leave-one-out MSE {loo_mse(X, y, 1e-3):.4f}")
print("Part 2 passed")

## Part 3 (coding task): The whole path from one decomposition

Write `loo_path(X, y, lams)`, which returns an array with the leave-one-out MSE for every
penalty in `lams`, using a single call to `np.linalg.svd(X, full_matrices=False)`. With
$X = U S V^\top$ and singular values $s_j$,

$$
H_\lambda = U \operatorname{diag}\!\Big(\frac{s_j^2}{s_j^2 + \lambda}\Big) U^\top ,
$$

so both the fitted values $H_\lambda y$ and the leverages come from $U$, $s$ and
$U^\top y$ for any $\lambda$. Then print the penalty with the lowest leave-one-out MSE.

In [ ]:
def loo_path(X, y, lams):
    """Leave-one-out MSE of ridge for every penalty in lams, from one SVD of X."""
    # TODO
    raise NotImplementedError

In [ ]:
path = np.asarray(loo_path(X, y, lams))
assert path.shape == lams.shape, f"expected one value per penalty, shape {lams.shape}, got {path.shape}"
one_at_a_time = np.array([loo_mse(X, y, lam) for lam in lams])
assert np.allclose(path, one_at_a_time, rtol=1e-5), (
    "the path differs from Part 2: the shrinkage factor is s^2 / (s^2 + lam), applied in the basis of U")
best = lams[np.argmin(path)]
for lam, v in zip(lams, path):
    print(f"lam {lam:8.0e}   leave-one-out MSE {v:.4f}")
print("lowest at lam =", best)
assert best not in (lams[0], lams[-1]), "the best penalty should not sit at the end of the grid here"
print("Part 3 passed")

## Solution

### Part 1: Leverages

In [ ]:
def leverages(X, lam):
    """Diagonal of X (X^T X + lam I)^{-1} X^T, shape (n,)."""
    M = X.T @ X + lam * np.identity(X.shape[1])
    A = np.linalg.solve(M, X.T)                     # M^{-1} X^T, shape p x n
    return np.sum(X * A.T, axis=1)

**Why it works.** `np.linalg.solve(M, X.T)` returns $M^{-1}X^\top$, whose column $i$ is $M^{-1}x_i$. Multiplying
it elementwise by row $i$ of $X$ and summing gives $x_i^\top M^{-1} x_i$ for every row in
$O(np^2)$ work and memory, against $O(n^2 p)$ for the full matrix. The largest leverages
belong to the points at the ends of $[-1, 1]$, where a high-degree polynomial has the most
freedom and each point pulls the curve toward itself.

### Part 2: Leave-one-out MSE from one fit

In [ ]:
def loo_mse(X, y, lam):
    """Exact leave-one-out MSE of ridge with penalty lam, from a single fit."""
    e = y - X @ ridge_fit(X, y, lam)
    return float(np.mean((e / (1 - leverages(X, lam))) ** 2))

**Why it works.** The residual $e_i$ comes from a fit that saw row $i$ and bent toward it. Dividing by
$1 - h_{ii}$ undoes exactly that pull, by the replacement argument in the written problem,
which holds for ridge because the penalty does not involve row $i$. The 30 refits of
`loo_by_refitting` collapse into one solve, and the two agree to rounding.

### Part 3: The whole path from one decomposition

In [ ]:
def loo_path(X, y, lams):
    """Leave-one-out MSE of ridge for every penalty in lams, from one SVD of X."""
    U, s, _ = np.linalg.svd(X, full_matrices=False)
    Uty = U.T @ y
    out = []
    for lam in lams:
        shrink = s ** 2 / (s ** 2 + lam)                 # eigenvalues of H_lambda
        e = y - U @ (shrink * Uty)
        h = (U ** 2) @ shrink                             # diagonal of U diag(shrink) U^T
        out.append(np.mean((e / (1 - h)) ** 2))
    return np.array(out)

**Why it works.** $X^\top X + \lambda I = V(S^2 + \lambda I)V^\top$, so
$H_\lambda = US(S^2 + \lambda I)^{-1}SU^\top$, a projection onto the columns of $U$ with
direction $j$ scaled by $s_j^2/(s_j^2 + \lambda)$. Its diagonal is the row sums of
$U^2$ weighted by those factors, and $H_\lambda y = U(\text{factors} \cdot U^\top y)$. After
the one decomposition each penalty costs $O(np)$, so a grid of hundreds of penalties is
as cheap as a single fit. The path is U-shaped: the smallest penalties leave the degree-12
fit free to chase noise at the ends, and the largest flatten the curve.